In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from scipy.stats import randint
from xgboost import XGBClassifier
from sklearn.svm import SVC



In [4]:
df = pd.read_csv(
    "../data/processed/clean_data.csv"
)

print(df.columns)

Index(['SeniorCitizen', 'tenure', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges',
       'Churn', 'num_services'],
      dtype='str')


In [5]:
X = df.drop("Churn",axis=1)
y = df["Churn"].map({
    "No":0,
    "Yes":1
})

X_train , X_test ,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
numeric = X_train.select_dtypes(
    include = ["int64","float64"]
).columns.tolist()

# print(numeric)


categorique = X_train.select_dtypes(
    include=["object"]
).columns.tolist()
# print(categorique)

C:\Users\errak\AppData\Local\Temp\ipykernel_18804\4088123566.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorique = X_train.select_dtypes(


In [7]:
prepocessor = ColumnTransformer(
    transformers=[(
            "num",StandardScaler(),numeric
        ),
        (
            "cat",OneHotEncoder(handle_unknown="ignore"),categorique
        )
    ]
)
X_train_prepared = prepocessor.fit_transform(X_train)

X_test_prepared = prepocessor.transform(X_test)


In [8]:
smote= SMOTE(random_state=42)

X_train_smote,y_train_smote = smote.fit_resample(X_train_prepared,y_train)

print(y_train.value_counts(normalize=True))
print(y_train_smote.value_counts(normalize=True))

Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Churn
0    0.5
1    0.5
Name: proportion, dtype: float64


### logostique regresion 

In [9]:
logistic_model = LogisticRegression(
    random_state=42
)

logistic_model.fit(
    X_train_smote,
    y_train_smote
)
y_pre_logistic = logistic_model.predict(X_test_prepared)

precision_logistic = precision_score(y_test,y_pre_logistic)
recall_logistic = recall_score(y_test,y_pre_logistic)
f1_logistic = f1_score(y_test,y_pre_logistic)
roc_auc_logistic = roc_auc_score(y_test,y_pre_logistic)
confusion_matrix_logistic = confusion_matrix(y_test,y_pre_logistic)




# print("precision : ",precision_logistic)
# print("recall : ",recall_logistic)
# print("f1 score : ",f1_logistic)
# print("roc_auc : ",roc_auc_logistic)
# print("confusion matrix : ",confusion_matrix_logistic)


# params 
# c =>force de regularisation
# penalty => typede regularisation 
# solver 
# max_iter => total d'iteration
# class_weight => pour equilibre des class 

#### opimisation logistique regresion 

In [10]:
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"],
    "max_iter": [1000]
}


random_search = RandomizedSearchCV(
    estimator=logistic_model,
    param_distributions=param_grid,
    n_iter=10,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

random_search.fit(
    X_train_smote,
    y_train_smote
)

print("Best parameters:")
print(random_search.best_params_)
best_model = random_search.best_estimator_

y_pre_logistic_optimise = best_model.predict(X_test_prepared)


precision_logistic_opt = precision_score(y_test,y_pre_logistic_optimise)
recall_logistic_opt = recall_score(y_test,y_pre_logistic_optimise)
f1_logistic_opt = f1_score(y_test,y_pre_logistic_optimise)
roc_auc_logistic_opt = roc_auc_score(y_test,y_pre_logistic_optimise)
confusion_matrix_logistic_opt = confusion_matrix(y_test,y_pre_logistic_optimise)




# print("precision : ",precision_logistic_opt)
# print("recall : ",recall_logistic_opt)
# print("f1 score : ",f1_logistic_opt)
# print("roc_auc : ",roc_auc_logistic_opt)
# print("confusion matrix : ",confusion_matrix_logistic_opt)



c:\Users\errak\Customer-Churn-Analysis-and-Prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\errak\Customer-Churn-Analysis-and-Prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Best parameters:
{'solver': 'liblinear', 'penalty': 'l1', 'max_iter': 1000, 'C': 100}


### decision three


In [11]:
dt = DecisionTreeClassifier(
    random_state=42
)

dt.fit(X_train_smote,y_train_smote)

dt_predict = dt.predict(X_test_prepared)

precision_dt = precision_score(y_test,dt_predict)
recall_dt = recall_score(y_test,dt_predict)
f1_dt = f1_score(y_test,dt_predict)
roc_auc_dt = roc_auc_score(y_test,dt_predict)
confusion_matrix_dt = confusion_matrix(y_test,dt_predict)


In [12]:
param_grid = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": randint(3, 20),
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    "max_features": [None, "sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

random_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=50,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

random_search.fit(
    X_train_smote,
    y_train_smote
)


best_model_dt = random_search.best_estimator_

dt_predict_opt = best_model_dt.predict(X_test_prepared)

dt_proba_opt = best_model_dt.predict_proba(X_test_prepared)[:, 1]

precision_dt_opt = precision_score(y_test,dt_predict_opt)
recall_dt_opt = recall_score(y_test,dt_predict_opt)
f1_dt_opt = f1_score(y_test,dt_predict_opt)
roc_auc_dt_opt = roc_auc_score(y_test,dt_proba_opt)
confusion_matrix_dt_opt = confusion_matrix(y_test,dt_predict_opt)


### Random Forest

In [13]:
random_model = RandomForestClassifier(
    random_state=42
)

random_model.fit(
    X_train_smote,
    y_train_smote
)

y_pred_rf = random_model.predict(X_test_prepared)


precision_random = precision_score(y_test,y_pred_rf)
recall_random = recall_score(y_test,y_pred_rf)
f1_random = f1_score(y_test,y_pred_rf)
roc_auc_random = roc_auc_score(y_test,y_pred_rf)
confusion_matrix_random = confusion_matrix(y_test,y_pred_rf)


In [14]:
param_dist = {
    "n_estimators": [100, 200, 300, 500, 800],
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None]
}

random_search = RandomizedSearchCV(
    estimator=random_model,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train_smote, y_train_smote)

best_model_random = random_search.best_estimator_
y_pre_random_opt = best_model.predict(X_test_prepared)



precision_random_opt = precision_score(y_test,y_pre_random_opt)
recall_random_opt = recall_score(y_test,y_pre_random_opt)
f1_random_opt = f1_score(y_test,y_pre_random_opt)
roc_auc_random_opt = roc_auc_score(y_test,y_pre_random_opt)
confusion_matrix_random_opt = confusion_matrix(y_test,y_pre_random_opt)


Fitting 5 folds for each of 30 candidates, totalling 150 fits


XGBoost


In [15]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb.fit(X_train_smote,y_train_smote)

y_pred_xgb = xgb.predict(X_test_prepared)

precision_xgboost = precision_score(y_test,y_pred_xgb)
recall_xgboost = recall_score(y_test,y_pred_xgb)
f1_xgboost = f1_score(y_test,y_pred_xgb)
roc_auc_xgboost = roc_auc_score(y_test,y_pred_xgb)
confusion_matrix_xgboost = confusion_matrix(y_test,y_pred_xgb)

In [16]:
param_dist_xgb = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [2, 3, 4, 5, 6],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.2, 0.5]
}

random_search_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=30,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search_xgb.fit(X_train_smote, y_train_smote)

y_pred_xgb_opt = random_search_xgb.predict(X_test_prepared)

precision_xgb_opt = precision_score(y_test,y_pred_xgb_opt)
recall_xgb_opt = recall_score(y_test,y_pred_xgb_opt)
f1_xgb_opt = f1_score(y_test,y_pred_xgb_opt)
roc_auc_xgb_opt = roc_auc_score(y_test,y_pred_xgb_opt)
confusion_matrix_xgb_opt = confusion_matrix(y_test,y_pred_xgb_opt)


Fitting 5 folds for each of 30 candidates, totalling 150 fits


In [17]:
evaluation = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Logistic Regression opt",
        "Random Foreast",
        "Random Foreast opt",
        "Decision three",
        "Decision three opt",
        "Xgboost",
        "Xgboost opt",
    ],
    "Precision": [
        precision_logistic,
        precision_logistic_opt,
        precision_random,
        precision_random_opt,
        precision_dt,
        precision_dt_opt,
        precision_xgboost,
        precision_xgb_opt,
    ],
    "Recall": [
        recall_logistic,
        recall_logistic_opt,
        recall_random,
        recall_random_opt,
        recall_dt,
        recall_dt_opt,
        recall_xgboost,
        recall_xgb_opt,
    ],
    "F1-score": [
        f1_logistic,
        f1_logistic_opt,
        f1_random,
        f1_random_opt,
        f1_dt,
        f1_dt_opt,
        f1_xgboost,
        f1_xgb_opt,
    ],
    "ROC-AUC": [
        roc_auc_logistic,
        roc_auc_logistic_opt,
        roc_auc_random,
        roc_auc_random_opt,
        roc_auc_dt,
        roc_auc_dt_opt,
        roc_auc_xgboost,
        roc_auc_xgb_opt,
    ],
    "TN": [
        confusion_matrix_logistic[0, 0],
        confusion_matrix_logistic_opt[0, 0],
        confusion_matrix_random[0,0],
        confusion_matrix_random_opt[0,0],
        confusion_matrix_dt[0,0],
        confusion_matrix_dt_opt[0,0],
        confusion_matrix_xgboost[0,0],
        confusion_matrix_xgb_opt[0,0],
    ],

    "FP": [
        confusion_matrix_logistic[0, 1],
        confusion_matrix_logistic_opt[0, 1],
        confusion_matrix_random[0,1],
        confusion_matrix_random_opt[0,1],
        confusion_matrix_dt[0,1],
        confusion_matrix_dt_opt[0,1],
        confusion_matrix_xgboost[0,1],
        confusion_matrix_xgb_opt[0,1],
    ],

    "FN": [
        confusion_matrix_logistic[1, 0],
        confusion_matrix_logistic_opt[1, 0],
        confusion_matrix_random[1,0],
        confusion_matrix_random_opt[1,0],
        confusion_matrix_dt[1,0],
        confusion_matrix_dt_opt[1,0],
        confusion_matrix_xgboost[1,0],
        confusion_matrix_xgb_opt[1,0],
    ],

    "TP": [
        confusion_matrix_logistic[1, 1],
        confusion_matrix_logistic_opt[1, 1],
        confusion_matrix_random[1,1],
        confusion_matrix_random_opt[1,1],
        confusion_matrix_dt[1,1],
        confusion_matrix_dt_opt[1,1],
        confusion_matrix_xgboost[1,1],
        confusion_matrix_xgb_opt[1,1],
    ]
})
evaluation[["Precision", "Recall", "F1-score", "ROC-AUC"]] = evaluation[["Precision", "Recall", "F1-score", "ROC-AUC"]].round(3)


evaluation.to_csv(
    "../data/model_evaluation/model_evaluation.csv",
    index=False
)